# Furniture taxonomy-enriched descriptions

Create a separate text variant for the same 20,000 furniture image-text pairs. The original dataset remains unchanged. Each enriched description begins with the complete furniture taxonomy after the constant `Home & Kitchen` root.

Example: `Furniture; Living Room Furniture; Tables; Coffee Tables. <original description>`

In [1]:
from pathlib import Path
import hashlib

from IPython.display import display
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SOURCE_PATH = PROJECT_ROOT / "data/meta_Home_and_Kitchen_furniture_siglip2_pairs_20k.csv"
ENRICHED_PATH = PROJECT_ROOT / "data/meta_Home_and_Kitchen_furniture_siglip2_pairs_20k_taxonomy_enriched.csv"
EXPECTED_ROWS = 20_000

assert SOURCE_PATH.is_file(), f"Missing source pairs: {SOURCE_PATH}"
source_sha256 = hashlib.sha256(SOURCE_PATH.read_bytes()).hexdigest()
print({"source": str(SOURCE_PATH), "sha256": source_sha256})

{'source': '/Users/vachemacbook/Desktop/RecSystem/RecSystem/data/meta_Home_and_Kitchen_furniture_siglip2_pairs_20k.csv', 'sha256': '99cab2b40e130ae721544718f125fe2dc6369e08ecb54705ccc7f228cdcfb408'}


In [2]:
source_df = pd.read_csv(SOURCE_PATH, dtype="string").fillna("")
required_columns = [
    "asin", "description", "furniture_subcategory", "category_path", "image_path"
]
missing_columns = set(required_columns) - set(source_df.columns)
assert not missing_columns, f"Missing columns: {sorted(missing_columns)}"
assert len(source_df) == EXPECTED_ROWS
assert source_df["asin"].is_unique
assert source_df["description"].str.strip().ne("").all()
assert source_df["category_path"].str.startswith("Home & Kitchen > Furniture").all()
assert source_df["image_path"].map(lambda value: (PROJECT_ROOT / value).is_file()).all()
print({
    "rows": len(source_df),
    "unique_asins": source_df["asin"].nunique(),
    "unique_category_paths": source_df["category_path"].nunique(),
})

{'rows': 20000, 'unique_asins': 20000, 'unique_category_paths': 142}


In [3]:
def category_components(category_path):
    return [component.strip() for component in category_path.split(">") if component.strip()]


def taxonomy_text(category_path):
    components = category_components(category_path)
    # Drop only the constant root. Keep Furniture and every more-specific label.
    informative_components = components[1:] if components[:1] == ["Home & Kitchen"] else components
    return "; ".join(informative_components)


enriched_df = source_df.rename(columns={"description": "description_original"}).copy()
enriched_df["taxonomy_text"] = enriched_df["category_path"].map(taxonomy_text)
enriched_df["description_enriched"] = (
    enriched_df["taxonomy_text"] + ". " + enriched_df["description_original"]
)
enriched_df = enriched_df[[
    "asin",
    "description_original",
    "taxonomy_text",
    "description_enriched",
    "furniture_subcategory",
    "category_path",
    "image_path",
]]

assert enriched_df["taxonomy_text"].str.startswith("Furniture").all()
assert all(
    enriched.endswith(original)
    for enriched, original in zip(
        enriched_df["description_enriched"], enriched_df["description_original"]
    )
)
assert enriched_df["description_enriched"].ne(enriched_df["description_original"]).all()
display(enriched_df.head(10))

,asin,description_original,taxonomy_text,description_enriched,furniture_subcategory,category_path,image_path
0,B018NZJ75M,"Offer a warm, inviting look to your kitchen, d...",Furniture; Game & Recreation Room Furniture; H...,Furniture; Game & Recreation Room Furniture; H...,Game & Recreation Room Furniture,Home & Kitchen > Furniture > Game & Recreation...,data/images_furniture/B018NZJ75M.jpg
1,B082XWFB66,Inspired by the playful style of the Italian M...,Furniture; Living Room Furniture; Ottomans,Furniture; Living Room Furniture; Ottomans. In...,Living Room Furniture,Home & Kitchen > Furniture > Living Room Furni...,data/images_furniture/B082XWFB66.jpg
2,B0892Q8NLR,Featuring classic design elements and plentifu...,"Furniture; Bedroom Furniture; Beds, Frames & B...","Furniture; Bedroom Furniture; Beds, Frames & B...",Bedroom Furniture,Home & Kitchen > Furniture > Bedroom Furniture...,data/images_furniture/B0892Q8NLR.jpg
3,B08D67VZFH,"Features: Made of premium birch, solid constru...",Furniture; Living Room Furniture; Chairs,Furniture; Living Room Furniture; Chairs. Feat...,Living Room Furniture,Home & Kitchen > Furniture > Living Room Furni...,data/images_furniture/B08D67VZFH.jpg
4,B0B35BM8RQ,"The 1.5-inch Bunkie Board , Split Bunkie Board...","Furniture; Bedroom Furniture; Beds, Frames & B...","Furniture; Bedroom Furniture; Beds, Frames & B...",Bedroom Furniture,Home & Kitchen > Furniture > Bedroom Furniture...,data/images_furniture/B0B35BM8RQ.jpg
5,B07V4PVRV1,Give your table a makeover with a new glass to...,Furniture; Dining Room Furniture; Tables,Furniture; Dining Room Furniture; Tables. Give...,Dining Room Furniture,Home & Kitchen > Furniture > Dining Room Furni...,data/images_furniture/B07V4PVRV1.jpg
6,B0088XR2WK,"Embracing a contemporary, youthful approach to...",Furniture; Game & Recreation Room Furniture; H...,Furniture; Game & Recreation Room Furniture; H...,Game & Recreation Room Furniture,Home & Kitchen > Furniture > Game & Recreation...,data/images_furniture/B0088XR2WK.jpg
7,B00BUY3AC0,The ultra handy and handsome perth espresso ov...,Furniture; Living Room Furniture; Tables; Coff...,Furniture; Living Room Furniture; Tables; Coff...,Living Room Furniture,Home & Kitchen > Furniture > Living Room Furni...,data/images_furniture/B00BUY3AC0.jpg
8,B00BQK2SN0,This Stanis circular end table is made of maho...,Furniture; Living Room Furniture; Tables; End ...,Furniture; Living Room Furniture; Tables; End ...,Living Room Furniture,Home & Kitchen > Furniture > Living Room Furni...,data/images_furniture/B00BQK2SN0.jpg
9,B00IECZHNQ,The Shape Chair has won various design prizes ...,Furniture; Living Room Furniture; Chairs,Furniture; Living Room Furniture; Chairs. The ...,Living Room Furniture,Home & Kitchen > Furniture > Living Room Furni...,data/images_furniture/B00IECZHNQ.jpg


In [4]:
enriched_df.to_csv(ENRICHED_PATH, index=False)
saved_df = pd.read_csv(ENRICHED_PATH, dtype="string").fillna("")

assert len(saved_df) == EXPECTED_ROWS
assert saved_df["asin"].is_unique
assert saved_df["description_original"].equals(enriched_df["description_original"])
assert saved_df["description_enriched"].equals(enriched_df["description_enriched"])
assert saved_df["image_path"].map(lambda value: (PROJECT_ROOT / value).is_file()).all()
assert hashlib.sha256(SOURCE_PATH.read_bytes()).hexdigest() == source_sha256

print({
    "saved_rows": len(saved_df),
    "unique_asins": saved_df["asin"].nunique(),
    "empty_original_descriptions": int(saved_df["description_original"].str.strip().eq("").sum()),
    "empty_enriched_descriptions": int(saved_df["description_enriched"].str.strip().eq("").sum()),
    "source_preserved": True,
    "output": str(ENRICHED_PATH),
})

{'saved_rows': 20000, 'unique_asins': 20000, 'empty_original_descriptions': 0, 'empty_enriched_descriptions': 0, 'source_preserved': True, 'output': '/Users/vachemacbook/Desktop/RecSystem/RecSystem/data/meta_Home_and_Kitchen_furniture_siglip2_pairs_20k_taxonomy_enriched.csv'}


## Experiment contract

The A/B model comparison must use identical ASIN splits, sampled rows, batch order, initialization, optimizer settings, and evaluation queries. Only the training text changes. Evaluate both models against original natural descriptions as the primary comparison; enriched-description evaluation is a secondary diagnostic because taxonomy will not necessarily be present in a real user query.